# AIoT ML Pipeline — Experiments Notebook
Generates all 11 figures and saves them to `report/figures/`.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')
ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
print('Root:', ROOT)

## 0. Generate dataset if needed

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, str(ROOT/'scripts'/'gen_data.py')], capture_output=True, text=True)
print(result.stdout or 'Dataset already present.')

## 1. Load & Preprocess

In [ ]:
from src.pipeline import load_raw, preprocess
df = load_raw(ROOT/'data'/'ai4i2020.csv')
X_train, X_test, y_train, y_test, scaler, df_proc, feat_names = preprocess(df)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Failure rate: {y_test.mean()*100:.2f}%')
df.head()

## Figure 1 — Class Distribution

In [ ]:
from src.viz import fig_class_distribution
fig = fig_class_distribution(df)
plt.show()

## Figure 2 — Correlation Heatmap

In [ ]:
from src.viz import fig_correlation_heatmap
fig = fig_correlation_heatmap(df)
plt.show()

## Figure 3 — Boxplots by Class

In [ ]:
from src.viz import fig_boxplots
fig = fig_boxplots(df)
plt.show()

## 2. Train Classifiers

In [ ]:
from src.models import train_all
fitted_models = train_all(X_train, y_train)
print('Trained:', list(fitted_models.keys()))

## Figure 4 — Model Comparison

In [ ]:
from src.evaluate import build_metrics_table
from src.viz import fig_model_comparison
metrics_df = build_metrics_table(fitted_models, X_test, y_test)
print(metrics_df.round(4))
fig = fig_model_comparison(metrics_df)
plt.show()

## Figure 5 — Confusion Matrix (Best Model)

In [ ]:
from src.evaluate import get_confusion_matrix
from src.viz import fig_confusion_matrix
best_name = metrics_df['F1'].idxmax()
cm = get_confusion_matrix(fitted_models[best_name], X_test, y_test)
print(f'Best model: {best_name}')
fig = fig_confusion_matrix(cm, best_name)
plt.show()

## Figure 6 — ROC Curves

In [ ]:
from src.evaluate import get_roc_data
from src.viz import fig_roc_curves
roc_data = get_roc_data(fitted_models, X_test, y_test)
fig = fig_roc_curves(roc_data)
plt.show()

## Figure 7 — Feature Importance + Decision Tree

In [ ]:
from src.viz import fig_feature_importance, fig_decision_tree
fig = fig_feature_importance(fitted_models['Random Forest'], feat_names)
plt.show()
fig = fig_decision_tree(fitted_models['Decision Tree'], feat_names)
plt.show()

## Figure 8 — Clustering PCA

In [ ]:
from src.evaluate import run_clustering
from src.viz import fig_clustering
clust_res = run_clustering(X_test)
for name, res in clust_res.items():
    sil = res['silhouette']
    print(f'{name}: silhouette = {sil:.4f}' if not (sil != sil) else f'{name}: silhouette = N/A')
fig = fig_clustering(clust_res)
plt.show()

## Figure 9 — DL Curves (ANN + LSTM)

In [ ]:
from src.dl_models import train_ann, train_lstm
from src.viz import fig_dl_curves
from sklearn.model_selection import train_test_split
Xtr, Xvl, ytr, yvl = train_test_split(X_train, y_train, test_size=0.125, random_state=42)
ann_model, ann_hist = train_ann(Xtr, ytr, Xvl, yvl, epochs=40)
lstm_model, lstm_hist = train_lstm(Xtr, ytr, Xvl, yvl, epochs=25)
fig = fig_dl_curves(ann_hist, lstm_hist)
plt.show()

## Figure 10 — Offloading Lab

In [ ]:
from src.offload_agent import run_offloading_simulation
from src.viz import fig_offloading_pie, fig_latency_histogram, fig_budget_met
sim = run_offloading_simulation(n_requests=500)
for policy, records in sim.items():
    pct = 100*np.mean([r['met_budget'] for r in records])
    print(f'{policy}: {pct:.1f}% budget met')
fig = fig_offloading_pie(sim, 'threshold'); plt.show()
fig = fig_latency_histogram(sim); plt.show()
fig = fig_budget_met(sim); plt.show()

## Figure 11 — A* Gantt

In [ ]:
import random
random.seed(42)
from src.offload_agent import AStarScheduler
from src.viz import fig_astar_gantt
tasks = [{'id': i+1, 'budget_ms': random.choice([40,60,80,120,200,300]), 'priority': random.randint(1,5)} for i in range(20)]
scheduler = AStarScheduler(tasks)
asgn = scheduler.schedule()
fig = fig_astar_gantt(asgn)
plt.show()
print('Tasks meeting budget:', sum(a['met_budget'] for a in asgn), '/', len(asgn))

## Done — All figures saved to report/figures/

In [ ]:
import os
fig_dir = ROOT/'report'/'figures'
figs = list(fig_dir.glob('*.png'))
print(f'Saved {len(figs)} figures:')
for f in sorted(figs):
    print(' ', f.name)